In [1]:
# Cell 1 — Imports

import pandas as pd                          # data manipulation and CSV reading
import os                                    # read environment variables (API keys)
import json                                  # format data cleanly before sending to LLM
import smtplib                               # send emails via Gmail SMTP
from email.mime.multipart import MIMEMultipart  # build multi-part email (plain + HTML)
from email.mime.text import MIMEText            # attach text/HTML content to the email
from groq import Groq                        # Groq SDK to call the LLaMA3 model
from datetime import datetime                # timestamp the report
from pathlib import Path                     # handle file paths cleanly across OS

print("All imports successful")

All imports successful


In [2]:
# Cell 2 — Load the anomaly report

df = pd.read_csv('../data/anomaly_report.csv', parse_dates=['date'])  # load the 25-row anomaly file, parse dates as datetime objects
df = df.sort_values('date').reset_index(drop=True)                    # sort chronologically, reset index to 0,1,2...

# quick sanity check
print(f"Loaded {len(df)} anomaly rows")                                           # confirms we have 25 rows
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")      # should show June 2022 to March 2024
print(f"\nAnomaly types:")                                                         # header line for the breakdown below
print(df['anomaly_type'].value_counts())                                           # shows split between Traffic Drop and Traffic Spike

Loaded 25 anomaly rows
Date range: 2022-06-10 to 2024-03-18

Anomaly types:
anomaly_type
Traffic Drop     21
Traffic Spike     4
Name: count, dtype: int64


In [3]:
# Cell 3 — Cluster consecutive anomaly days into incidents

clusters = []   # will hold each completed incident cluster
current = None  # tracks the cluster we are currently building

for _, row in df.iterrows():                                              # loop through every anomaly row one by one
    if current is None:                                                   # first row — start a new cluster
        current = {
            'anomaly_type': row['anomaly_type'],                          # track the type so we don't mix drops and spikes
            'start_date':   row['date'],                                  # first day of this incident
            'end_date':     row['date'],                                  # will update as we add more days
            'days':         [row]                                         # list of individual rows in this cluster
        }
    else:
        gap       = (row['date'] - current['end_date']).days             # days between this row and the last one in current cluster
        same_type = row['anomaly_type'] == current['anomaly_type']       # check they are the same anomaly type

        if same_type and gap <= 3:                                        # close enough and same type — extend current cluster
            current['end_date'] = row['date']                            # push the end date forward
            current['days'].append(row)                                  # add this day to the cluster
        else:                                                             # gap too large or different type — save and start fresh
            clusters.append(current)
            current = {
                'anomaly_type': row['anomaly_type'],
                'start_date':   row['date'],
                'end_date':     row['date'],
                'days':         [row]
            }

if current:                                                               # don't forget the last cluster after the loop ends
    clusters.append(current)

# summarise each cluster into a single stats dictionary
summaries = []
for c in clusters:
    days_df = pd.DataFrame(c['days'])                                    # convert list of rows into a small DataFrame for easy maths
    summaries.append({
        'anomaly_type':        c['anomaly_type'],
        'start_date':          c['start_date'].strftime('%Y-%m-%d'),     # format as clean string for the LLM prompt
        'end_date':            c['end_date'].strftime('%Y-%m-%d'),
        'duration_days':       len(days_df),                             # how many days this incident lasted
        'avg_actual':          int(days_df['actual_sessions'].mean()),   # average real sessions during the incident
        'avg_predicted':       int(days_df['predicted_sessions'].mean()), # what Prophet expected on those days
        'avg_deviation_pct':   round(days_df['deviation_pct'].mean(), 1), # average % gap between actual and predicted
        'peak_deviation_pct':  round(days_df['deviation_pct'].abs().max(), 1), # worst single day deviation
        'total_session_delta': int(days_df['deviation'].sum())           # total sessions gained or lost across the incident
    })

# preview what we built
print(f"Found {len(summaries)} incident cluster(s):\n")
for s in summaries:                                                       # print a one-line summary per cluster
    print(f"  [{s['anomaly_type']}] {s['start_date']} to {s['end_date']} "
          f"({s['duration_days']} days) | avg deviation: {s['avg_deviation_pct']}%")

Found 2 incident cluster(s):

  [Traffic Drop] 2022-06-10 to 2022-06-30 (21 days) | avg deviation: -52.5%
  [Traffic Spike] 2024-03-15 to 2024-03-18 (4 days) | avg deviation: 232.2%


In [4]:
# Cell 4 — Connect to Groq and define the narrative generator

client = Groq(api_key=os.environ.get('GROQ_API_KEY'))    # initialise Groq client using API key from environment variable

def generate_narrative(cluster_summary: dict) -> str:
    """
    Takes one cluster summary dictionary and returns a plain-English
    analyst narrative paragraph from the Groq LLaMA3 model.
    """

    prompt = f"""
You are a senior digital marketing analyst. Write a concise incident summary
for the following anomaly detected in website traffic data.

Incident data:
{json.dumps(cluster_summary, indent=2)}

Instructions:
- Write exactly 3 sentences.
- Sentence 1: What happened (dates, duration, direction, magnitude).
- Sentence 2: What this pattern most likely indicates in the real world
  (Traffic Drop sustained over weeks = likely algorithm update or technical SEO issue;
   Traffic Spike over a few days = likely bot traffic or tracking pixel misfire).
- Sentence 3: Recommended immediate investigation step.
- Use plain business English. No bullet points. No markdown. No preamble.
- Do not start with 'I' or 'The incident'.
"""                                                       # structured prompt that tells the LLM exactly what to write and how

    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',                           # fast, free-tier Groq model — sufficient for business writing
        messages=[{'role': 'user', 'content': prompt}],  # standard chat format — one user message containing our prompt
        temperature=0.4,                                  # low temperature = consistent professional tone, not creative
        max_tokens=300                                    # 3 sentences needs at most ~150 tokens, 300 gives comfortable headroom
    )

    return response.choices[0].message.content.strip()   # extract the text from the response and remove leading/trailing whitespace

print("Groq client ready")                               # confirms the client initialised without errors

Groq client ready


In [5]:
# Cell 5 — Generate a narrative for each cluster

for i, summary in enumerate(summaries):                                        # loop through each incident cluster with an index
    print(f"Generating narrative {i+1}/{len(summaries)}: "
          f"{summary['anomaly_type']} ({summary['start_date']})...")           # show progress so we know which cluster is being processed
    summary['narrative'] = generate_narrative(summary)                         # call the LLM and store the result back in the summary dict
    print(f"  Done\n")                                                         # confirm this cluster is complete before moving to the next

# preview the narratives
for s in summaries:                                                            # loop through all summaries to print the results
    print(f"=== {s['anomaly_type']} | {s['start_date']} to {s['end_date']} ===")  # header showing which incident this narrative is for
    print(s['narrative'])                                                      # print the LLM generated paragraph
    print()                                                                    # blank line between clusters for readability

Generating narrative 1/2: Traffic Drop (2022-06-10)...
  Done

Generating narrative 2/2: Traffic Spike (2024-03-15)...
  Done

=== Traffic Drop | 2022-06-10 to 2022-06-30 ===
Website traffic experienced a significant drop from June 10, 2022, to June 30, 2022, lasting 21 days, with an average deviation of 52.5% below predicted levels, resulting in a total session delta of -135689. This sustained traffic drop over several weeks likely indicates a technical SEO issue or a major algorithm update that negatively impacted the website's visibility and ranking. Immediate investigation should focus on reviewing recent website changes, SEO audits, and search engine algorithm updates to identify the root cause of the traffic drop and develop a corrective action plan.

=== Traffic Spike | 2024-03-15 to 2024-03-18 ===
A traffic spike occurred from March 15 to March 18, 2024, lasting 4 days, with an average actual traffic of 58,889, which is 232.2% higher than the predicted average, resulting in a t

In [6]:
# Cell 6 — Build the HTML email report

def build_html_report(summaries: list, generated_at: str) -> str:      # takes our list of summaries and a timestamp string
    total_incidents    = len(summaries)                                 # total number of incident clusters
    total_anomaly_days = sum(s['duration_days'] for s in summaries)    # total days flagged across all clusters
    drops  = [s for s in summaries if s['anomaly_type'] == 'Traffic Drop']   # filter to just drop incidents
    spikes = [s for s in summaries if s['anomaly_type'] == 'Traffic Spike']  # filter to just spike incidents

    # colour scheme per anomaly type — red for drops, orange for spikes
    TYPE_COLOURS = {
        'Traffic Drop':  {'border': '#E53E3E', 'badge_bg': '#E53E3E'},
        'Traffic Spike': {'border': '#DD6B20', 'badge_bg': '#DD6B20'},
    }

    def incident_card(s):                                               # builds one HTML card block per incident
        c         = TYPE_COLOURS.get(s['anomaly_type'])                # get the colour config for this type
        direction = '▼' if s['avg_deviation_pct'] < 0 else '▲'       # down arrow for drops, up arrow for spikes
        deviation = f"{direction} {abs(s['avg_deviation_pct'])}% avg deviation"  # e.g. ▼ 52.5% avg deviation
        delta     = f"{s['total_session_delta']:+,} sessions total"   # e.g. -135,689 sessions total

        return f"""
        <table width="100%" cellpadding="0" cellspacing="0" style="margin-bottom:20px; border:1px solid {c['border']}; border-radius:8px;">
          <tr>
            <td style="padding:20px;">
              <span style="background:{c['badge_bg']}; color:#fff; font-size:11px; font-weight:700; padding:3px 10px; border-radius:20px;">{s['anomaly_type']}</span>
              <p style="margin:10px 0 4px; font-size:16px; font-weight:700; font-family:Arial,sans-serif;">{s['start_date']} — {s['end_date']}</p>
              <p style="margin:0; font-size:13px; color:#4A5568; font-family:Arial,sans-serif;">{s['duration_days']} days &nbsp;|&nbsp; {deviation} &nbsp;|&nbsp; {delta}</p>
              <p style="margin:14px 0 0; font-size:14px; line-height:1.7; color:#2D3748; font-family:Georgia,serif; font-style:italic; border-left:3px solid {c['border']}; padding-left:14px;">{s['narrative']}</p>
            </td>
          </tr>
        </table>
        """                                                             # each card is a self-contained HTML table block

    cards_html = '\n'.join(incident_card(s) for s in summaries)       # join all cards into one HTML string

    html = f"""
<!DOCTYPE html>
<html lang="en">
<head><meta charset="UTF-8"><title>Traffic Anomaly Report</title></head>
<body style="margin:0; padding:0; background:#F7FAFC; font-family:Arial,sans-serif;">

  <table width="100%" cellpadding="0" cellspacing="0" style="padding:30px 0;">
    <tr><td align="center">
      <table width="600" cellpadding="0" cellspacing="0">

        <!-- HEADER -->
        <tr>
          <td style="background:#1A202C; border-radius:10px 10px 0 0; padding:30px;">
            <p style="margin:0; font-size:11px; color:#A0AEC0; text-transform:uppercase; letter-spacing:1px;">Automated Analytics Report</p>
            <h1 style="margin:8px 0 4px; font-size:24px; color:#FFFFFF; font-family:Arial,sans-serif;">Traffic Anomaly Summary</h1>
            <p style="margin:0; font-size:13px; color:#A0AEC0;">Generated {generated_at}</p>
          </td>
        </tr>

        <!-- STATS BAR -->
        <tr>
          <td style="background:#2D3748; padding:16px 30px;">
            <table width="100%" cellpadding="0" cellspacing="0">
              <tr>
                <td style="text-align:center;">
                  <p style="margin:0; font-size:28px; font-weight:700; color:#FFFFFF;">{total_incidents}</p>
                  <p style="margin:2px 0 0; font-size:11px; color:#A0AEC0; text-transform:uppercase;">Incidents</p>
                </td>
                <td style="text-align:center;">
                  <p style="margin:0; font-size:28px; font-weight:700; color:#FFFFFF;">{total_anomaly_days}</p>
                  <p style="margin:2px 0 0; font-size:11px; color:#A0AEC0; text-transform:uppercase;">Anomaly Days</p>
                </td>
                <td style="text-align:center;">
                  <p style="margin:0; font-size:28px; font-weight:700; color:#FC8181;">{len(drops)}</p>
                  <p style="margin:2px 0 0; font-size:11px; color:#A0AEC0; text-transform:uppercase;">Traffic Drops</p>
                </td>
                <td style="text-align:center;">
                  <p style="margin:0; font-size:28px; font-weight:700; color:#F6AD55;">{len(spikes)}</p>
                  <p style="margin:2px 0 0; font-size:11px; color:#A0AEC0; text-transform:uppercase;">Traffic Spikes</p>
                </td>
              </tr>
            </table>
          </td>
        </tr>

        <!-- INCIDENT CARDS -->
        <tr>
          <td style="background:#FFFFFF; padding:30px; border-radius:0 0 10px 10px;">
            <h2 style="margin:0 0 20px; font-size:15px; color:#4A5568; text-transform:uppercase; letter-spacing:0.5px;">Incident Details</h2>
            {cards_html}
          </td>
        </tr>

        <!-- FOOTER -->
        <tr>
          <td style="padding:20px 0; text-align:center;">
            <p style="margin:0; font-size:12px; color:#A0AEC0;">Anomaly detection by Facebook Prophet &nbsp;·&nbsp; Narratives by Groq LLaMA3</p>
            <p style="margin:4px 0 0; font-size:12px; color:#A0AEC0;">This report was generated automatically. Do not reply to this email.</p>
          </td>
        </tr>

      </table>
    </td></tr>
  </table>

</body>
</html>
"""                                                                     # end of the full HTML document string
    return html                                                         # return the complete HTML as a string

generated_at = datetime.now().strftime('%B %d, %Y at %H:%M UTC')      # e.g. "June 28, 2026 at 09:45 UTC"
html_report  = build_html_report(summaries, generated_at)             # call the function to build the report

print(f"HTML report built — {len(html_report):,} characters")         # confirm the report was created and show its size

HTML report built — 6,086 characters


In [7]:
# Cell 7 — Save the HTML report to disk

report_date = datetime.now().strftime('%Y-%m-%d')                              # e.g. "2026-06-28" — used in the filename
report_path = Path(f'../data/anomaly_report_{report_date}.html')               # full path: data/anomaly_report_2026-06-28.html

with open(report_path, 'w', encoding='utf-8') as f:                            # open the file for writing in UTF-8 encoding
    f.write(html_report)                                                        # write the full HTML string to disk

print(f"Report saved to: {report_path}")                                        # confirm the file was saved and show the path

Report saved to: ../data/anomaly_report_2026-06-28.html


In [9]:
# Cell 8 — Send the report via Gmail SMTP

GMAIL_USER = os.environ.get('GMAIL_USER')                             # sender Gmail address from environment variable
GMAIL_PASS = os.environ.get('GMAIL_PASS')                             # Gmail App Password from environment variable
REPORT_TO  = os.environ.get('REPORT_TO', GMAIL_USER)                 # recipient address — defaults to sender if not set

if not GMAIL_USER or not GMAIL_PASS:                                  # guard clause — skip sending if credentials are missing
    print("Email credentials not set. Skipping send.")
    print("Set GMAIL_USER and GMAIL_PASS environment variables to enable.")
else:
    subject = f"Traffic Anomaly Report — {len(summaries)} Incident(s) Detected — {report_date}"  # email subject line

    msg = MIMEMultipart('alternative')                                # create a multi-part email container
    msg['Subject'] = subject                                          # attach the subject line
    msg['From']    = GMAIL_USER                                       # sender address
    msg['To']      = REPORT_TO                                        # recipient address

    # plain text fallback — shown if the recipient's email client can't render HTML
    plain_lines = [f"Traffic Anomaly Report — {report_date}", ""]
    for s in summaries:                                               # one block of text per incident
        plain_lines.append(f"{s['anomaly_type']}: {s['start_date']} to {s['end_date']}")
        plain_lines.append(f"  Duration: {s['duration_days']} days")
        plain_lines.append(f"  Avg deviation: {s['avg_deviation_pct']}%")
        plain_lines.append(f"  {s['narrative']}")
        plain_lines.append("")
    plain_text = '\n'.join(plain_lines)                               # join all lines into a single plain text string

    msg.attach(MIMEText(plain_text, 'plain'))                         # attach plain text version first
    msg.attach(MIMEText(html_report, 'html'))                         # attach HTML version second — email clients prefer this one

    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:       # connect to Gmail's SMTP server over SSL on port 465
            server.login(GMAIL_USER, GMAIL_PASS)                      # authenticate with your credentials
            server.sendmail(GMAIL_USER, REPORT_TO, msg.as_string())   # send the email
        print(f"Email sent to {REPORT_TO}")                           # confirm successful delivery
    except Exception as e:
        print(f"Email failed: {e}")                                   # print the error if something goes wrong

Email sent to priyank.work09@gmail.com


In [10]:
# Cell 9 — Final summary

print('=' * 55)                                                        # top border
print('DAY 3 NOTEBOOK COMPLETE')
print('=' * 55)
print(f'  Anomaly clusters processed : {len(summaries)}')             # how many incident clusters were analysed
print(f'  LLM narratives generated   : {len(summaries)}')             # one narrative per cluster
print(f'  HTML report saved          : {report_path}')                # path to the saved HTML file
print(f'  Email delivered to         : {REPORT_TO}')                  # who received the report
print('=' * 55)

DAY 3 NOTEBOOK COMPLETE
  Anomaly clusters processed : 2
  LLM narratives generated   : 2
  HTML report saved          : ../data/anomaly_report_2026-06-28.html
  Email delivered to         : priyank.work09@gmail.com
